# Глобальное сравнение IT: США / Россия / ЕС (2025/2026)

## Единая валюта (USD), поправка на покупательную способность (PPP) и грейды

Сводный проект, объединяющий три региональных анализа (проекты 18 — США, 19 — Россия, 20 — ЕС) в **единой методологии**:

1. Все зарплаты и расходы приведены к **долларам США** по рыночному курсу 2025.
2. Дополнительно считается **реальная покупательная способность (PPP)** — поправка на уровень цен (Price Level Index, США = 100).
3. Учитываются **грейды**: junior / middle / senior (множители к средней зарплате).

### Зачем PPP?
$60\,000 в Люксембурге и $60\,000 в Казани — это **разный уровень жизни**, потому что цены различаются. PPP переводит номинальный доход в «международные доллары» (int$), сопоставимые по покупательной способности.

$$\text{Реальный доход (int\$)} = \frac{\text{Номинальный доход (USD)}}{\text{PLI}/100}$$

### Содержание:
1. Единый набор данных (репрезентативные локации трёх регионов)
2. Приведение к USD и PPP
3. Грейды junior / middle / senior
4. Сравнение регионов
5. Глобальный рейтинг (номинал vs PPP)
6. Чувствительность к грейду
7. Выводы

> **Оговорка:** курсы валют, уровни цен (PLI) и множители грейдов — приблизительные оценки 2025 года. Определение «на руки» единое: доход минус подоходный налог и **взносы работника** (взносы работодателя исключены во всех регионах для сопоставимости).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 40)
print('Библиотеки загружены!')

## 1. Единый набор данных

Репрезентативные локации из трёх региональных проектов. Для каждой:
- `gross_local` — средняя годовая зарплата **middle**-разработчика (местная валюта);
- `ded_rate` — доля удержаний с работника (подоходный налог + соц. взносы работника);
- `rent_local` — аренда 1-комн. в центре (местная валюта, в месяц);
- `PLI` — индекс уровня цен (США = 100), для поправки на PPP.

**Определение удержаний по регионам (для сопоставимости):**
- США: федеральный + штатный подоходный налог + FICA (соц. взносы работника ~7,65%);
- Россия: только НДФЛ 13% (соц. взносы платит работодатель сверх оклада);
- ЕС: подоходный налог + соц. взносы работника (как в проекте 20).

In [ ]:
# region, location, currency, gross_local(год), ded_rate, rent_local(мес), PLI(US=100)
ROWS = [
    # --- США (USD), ded = fed+state подоходный + FICA 7.65% ---
    ('США', 'Вашингтон (WA)',      'USD',   158000, 0.25, 1650, 108),
    ('США', 'Техас (TX)',          'USD',   130000, 0.24, 1350,  93),
    ('США', 'Калифорния (CA)',     'USD',   159000, 0.34, 1900, 135),
    ('США', 'Нью-Йорк (NY)',       'USD',   137000, 0.31, 1550, 125),
    # --- Россия (RUB), ded = НДФЛ 13% ---
    ('Россия', 'Москва',           'RUB',  3000000, 0.13, 60000,  45),
    ('Россия', 'Новосибирск',      'RUB',  2160000, 0.13, 28000,  33),
    ('Россия', 'Казань',           'RUB',  2100000, 0.13, 30000,  33),
    # --- ЕС (EUR), ded = подоходный + соц. взносы работника ---
    ('ЕС', 'Люксембург',           'EUR',    85000, 0.29, 1800, 110),
    ('ЕС', 'Германия',             'EUR',    68000, 0.39, 1250, 100),
    ('ЕС', 'Нидерланды',           'EUR',    66000, 0.32, 1900, 108),
    ('ЕС', 'Ирландия',             'EUR',    70000, 0.28, 2100, 125),
    ('ЕС', 'Эстония',              'EUR',    42000, 0.21,  800,  75),
    ('ЕС', 'Польша',               'EUR',    40000, 0.24,  950,  58),
    ('ЕС', 'Португалия',           'EUR',    35000, 0.29, 1300,  70),
]
df = pd.DataFrame(ROWS, columns=['region', 'location', 'currency',
                                 'gross_local', 'ded_rate', 'rent_local', 'PLI'])

# Рыночные курсы 2025 (к USD)
FX = {'USD': 1.0, 'EUR': 1.08, 'RUB': 1 / 92.0}
df['fx'] = df['currency'].map(FX)
print(f'Локаций: {len(df)} | Регионы: {df["region"].unique().tolist()}')
print(f'Курсы к USD: {FX}')
print(df.head())

## 2. Приведение к USD и PPP

Переводим зарплату и аренду в USD, считаем доход «на руки» и располагаемый доход (после аренды), затем корректируем на уровень цен (PPP → int$).

In [ ]:
# Номинал в USD
df['gross_usd'] = df['gross_local'] * df['fx']
df['net_usd'] = df['gross_usd'] * (1 - df['ded_rate'])
df['rent_usd_year'] = df['rent_local'] * 12 * df['fx']
df['dispo_usd'] = df['net_usd'] - df['rent_usd_year']

# PPP: реальная покупательная способность (int$)
df['net_ppp'] = df['net_usd'] / (df['PLI'] / 100)
df['dispo_ppp'] = df['dispo_usd'] / (df['PLI'] / 100)

show = df[['region', 'location', 'gross_usd', 'net_usd', 'dispo_usd',
           'dispo_ppp']].sort_values('dispo_ppp', ascending=False).copy()
for c in ['gross_usd', 'net_usd', 'dispo_usd', 'dispo_ppp']:
    show[c] = show[c].map('${:,.0f}'.format)
print('Номинал (USD) vs реальная покупательная способность (int$)')
print('=' * 80)
print(show.to_string(index=False))

In [ ]:
# Номинал vs PPP: как меняется картина
srt = df.sort_values('dispo_ppp', ascending=True)
region_color = {'США': '#4C72B0', 'Россия': '#C44E52', 'ЕС': '#55A868'}
colors = srt['region'].map(region_color)

fig, axes = plt.subplots(1, 2, figsize=(17, 8))
axes[0].barh(srt['location'], srt['dispo_usd'] / 1000, color=colors, alpha=0.85)
axes[0].set_xlabel('Располагаемый доход, номинал (тыс. USD/год)')
axes[0].set_title('Номинальный располагаемый доход (рыночный курс)')

axes[1].barh(srt['location'], srt['dispo_ppp'] / 1000, color=colors, alpha=0.85)
axes[1].set_xlabel('Располагаемый доход, PPP (тыс. int$/год)')
axes[1].set_title('Реальная покупательная способность (PPP)')

from matplotlib.patches import Patch
legend = [Patch(facecolor=c, label=r) for r, c in region_color.items()]
axes[1].legend(handles=legend, loc='lower right')
plt.tight_layout()
plt.show()

print('Вывод: PPP подтягивает Россию и восточную Европу (низкие цены),')
print('и снижает дорогие локации (CA, NY, Ирландия, Люксембург).')

## 3. Грейды: junior / middle / senior

Зарплаты в наборе данных — для уровня **middle**. Применим типичные множители (относительно middle):

| Грейд | Множитель |
|---|---|
| junior | 0.60 |
| middle | 1.00 |
| senior | 1.60 |

> Множители усреднённые; в реальности senior-премия в США выше, чем в ЕС/РФ. Аренда и налоговая ставка при смене грейда упрощённо считаются неизменными (налоговая ставка де-факто растёт для senior из-за прогрессии — см. упражнения).

In [ ]:
GRADES = {'junior': 0.60, 'middle': 1.00, 'senior': 1.60}

def grade_frame(data, mult):
    g = data.copy()
    g['gross_usd'] = g['gross_usd'] * mult
    g['net_usd'] = g['gross_usd'] * (1 - g['ded_rate'])
    g['dispo_usd'] = g['net_usd'] - g['rent_usd_year']
    g['dispo_ppp'] = g['dispo_usd'] / (g['PLI'] / 100)
    return g

graded = {name: grade_frame(df, m) for name, m in GRADES.items()}

# Средний реальный располагаемый доход (PPP) по регионам и грейдам
summary = pd.DataFrame({
    name: g.groupby('region')['dispo_ppp'].mean() for name, g in graded.items()
})
summary = summary.loc[['США', 'ЕС', 'Россия']]
print('Средний располагаемый доход по регионам (PPP, int$/год)')
print('=' * 60)
print(summary.map('${:,.0f}'.format).to_string())

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
x = np.arange(len(summary.index))
w = 0.25
for i, grade in enumerate(['junior', 'middle', 'senior']):
    ax.bar(x + (i - 1) * w, summary[grade] / 1000, w, label=grade, alpha=0.9)
ax.set_xticks(x)
ax.set_xticklabels(summary.index)
ax.set_ylabel('Располагаемый доход (тыс. int$/год, PPP)')
ax.set_title('Реальный располагаемый доход по регионам и грейдам')
ax.legend(title='Грейд')
for i, grade in enumerate(['junior', 'middle', 'senior']):
    for j, v in enumerate(summary[grade]):
        ax.text(j + (i - 1) * w, v / 1000 + 0.3, f'{v/1000:.0f}',
                ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## 4–5. Глобальный рейтинг локаций

Ранжируем все локации по **реальному располагаемому доходу (PPP)** для выбранного грейда.

In [ ]:
GRADE = 'middle'   # можно поменять на 'junior' / 'senior'
g = graded[GRADE].sort_values('dispo_ppp', ascending=False).reset_index(drop=True)
g['rank'] = g.index + 1

print(f'ГЛОБАЛЬНЫЙ РЕЙТИНГ ({GRADE}), по реальному располагаемому доходу (PPP)')
print('=' * 90)
view = g[['rank', 'region', 'location', 'gross_usd', 'net_usd',
          'dispo_usd', 'dispo_ppp']].copy()
for c in ['gross_usd', 'net_usd', 'dispo_usd', 'dispo_ppp']:
    view[c] = view[c].map('${:,.0f}'.format)
print(view.to_string(index=False))

In [ ]:
# Номинал vs PPP: стрелки изменения ранга для наглядности
nom = df.assign(dispo=grade_frame(df, GRADES[GRADE])['dispo_usd'])
nom_rank = nom.sort_values('dispo', ascending=False).reset_index(drop=True)
nom_rank['nom_rank'] = nom_rank.index + 1
ppp_rank = g[['location', 'rank']].rename(columns={'rank': 'ppp_rank'})
cmp = nom_rank[['location', 'region', 'nom_rank']].merge(ppp_rank, on='location')
cmp = cmp.sort_values('ppp_rank')

fig, ax = plt.subplots(figsize=(10, 8))
for _, r in cmp.iterrows():
    ax.plot([0, 1], [r['nom_rank'], r['ppp_rank']], '-o',
            color=region_color[r['region']], alpha=0.7)
    ax.text(-0.03, r['nom_rank'], r['location'], ha='right', va='center', fontsize=8)
    ax.text(1.03, r['ppp_rank'], r['location'], ha='left', va='center', fontsize=8)
ax.set_xticks([0, 1])
ax.set_xticklabels(['Ранг (номинал USD)', 'Ранг (PPP int$)'])
ax.invert_yaxis()
ax.set_title(f'Изменение ранга при переходе номинал → PPP ({GRADE})')
ax.set_xlim(-0.4, 1.4)
plt.tight_layout()
plt.show()

## 6. Чувствительность к грейду

Как меняется топ-локация при переходе junior → middle → senior? Senior сильнее выигрывает там, где выше номинальные зарплаты (США).

In [ ]:
ranks = pd.DataFrame({'location': df['location'], 'region': df['region']})
for name, gr in graded.items():
    r = gr.sort_values('dispo_ppp', ascending=False).reset_index()
    order = {loc: i + 1 for i, loc in enumerate(r['location'])}
    ranks[name] = ranks['location'].map(order)

top = ranks.sort_values('middle').head(8)
print('Ранг локации (PPP) по грейдам — топ-8 по middle')
print('=' * 55)
print(top.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
for _, r in top.iterrows():
    ax.plot(['junior', 'middle', 'senior'],
            [r['junior'], r['middle'], r['senior']],
            marker='o', color=region_color[r['region']], label=r['location'])
ax.invert_yaxis()
ax.set_ylabel('Ранг (1 = лучший, PPP)')
ax.set_title('Изменение рейтинга локации по грейдам')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 7. Выводы

In [ ]:
print('ГЛОБАЛЬНЫЕ ВЫВОДЫ (США / Россия / ЕС, 2025/2026)')
print('=' * 70)
for grade in ['junior', 'middle', 'senior']:
    gg = graded[grade]
    best_nom = gg.loc[gg['dispo_usd'].idxmax()]
    best_ppp = gg.loc[gg['dispo_ppp'].idxmax()]
    print(f'\n[{grade.upper()}]')
    print(f'  Номинал (USD):  {best_nom["location"]:<18} '
          f'${best_nom["dispo_usd"]:,.0f}/год')
    print(f'  Реальный (PPP): {best_ppp["location"]:<18} '
          f'${best_ppp["dispo_ppp"]:,.0f} int$/год')
print('\n' + '-' * 70)
print('КЛЮЧЕВЫЕ НАБЛЮДЕНИЯ:')
print('  1. По номиналу (USD) лидируют США — самые высокие зарплаты.')
print('  2. По PPP разрыв сокращается: низкие цены в РФ и Вост. Европе')
print('     повышают реальную покупательную способность.')
print('  3. Senior сильнее выигрывает в США (высокая номинальная база).')
print('  4. Дорогие локации (CA, NY, Ирландия, Люксембург) теряют позиции')
print('     при переходе к PPP из-за высокого уровня цен.')
print('  5. Выбор зависит от цели: накопить в валюте (США номинал) или')
print('     максимум комфорта локально (PPP-лидеры).')

## Источники и оговорки

**Данные** — снимки 2025 из проектов 18 (США), 19 (Россия), 20 (ЕС): BLS OEWS, Census ACS, hh.ru, Habr Career, ЦИАН, Stack Overflow Survey, Numbeo, OECD Taxing Wages.

**Оговорки:**
1. **Курсы валют** (EUR 1.08, RUB 1/92) — на 2025; волатильны и сильно влияют на номинал.
2. **PLI (уровень цен)** — приблизительные значения относительно США=100 (на базе World Bank ICP / Numbeo COL); реальный PPP по стране/городу отличается.
3. **Множители грейдов** — усреднённые; фактический разброс зависит от стека, компании, страны.
4. **Определение удержаний** унифицировано (налог + взносы работника), но базы «gross» в регионах определяются немного по-разному (см. раздел 1).
5. Не учтены: медстраховка в США (часто от работодателя), НДС, бонусы/опционы, стоимость релокации и визовые ограничения.

## Упражнения

### Упражнение 1: Прогрессивный налог по грейдам
Senior получает больше — и попадает в более высокие налоговые ставки. Замените фиксированную `ded_rate` на прогрессивную функцию (хотя бы для США и РФ) и пересчитайте рейтинг senior.

### Упражнение 2: Своя валюта накоплений
Добавьте цель «накопить $X за 3 года». Ранжируйте по номинальному располагаемому доходу в USD (без PPP) — как меняется топ?

### Упражнение 3: Чувствительность к курсу
Проварьируйте курс RUB (70, 92, 110) и EUR (1.0, 1.08, 1.2). Насколько устойчив рейтинг?

### Упражнение 4: Стоимость релокации
Добавьте единовременные затраты на переезд и «налоговый клин» первого года. Через сколько лет окупается переезд в топ-локацию?

### Упражнение 5: Свои веса и грейд
Постройте личный индекс: задайте свой грейд, важность налогов/аренды/накоплений и получите персональный топ-5.

---

**Решения** можно найти в ноутбуке `solutions/21_Solutions.ipynb`